In [86]:
import sys
sys.path.append("./starter-kit")
import openpyxl as pxl
import pandas as pd 
import matplotlib.pyplot as plt
import time
from datetime import datetime

from validation_structure import normaliser, valider 
pd.set_option("display.max_rows", None)

df = pd.read_csv("./starter-kit/data/numeros_tva.csv", dtype=str, keep_default_na=False)

In [87]:
## fonction normaliser 
normaliser("lu71043234")

'LU71043234'

In [88]:
# fonction valider 
valider("PT78732723")

(False, 'cle_invalide')

In [89]:
df["numero_normalise_tva"] = df["numero_tva"].apply(normaliser)
df.head(5)

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_normalise_tva
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88


In [90]:
df["numero_valider_tva"] = df["numero_normalise_tva"].apply(valider)
df.head(5)

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_normalise_tva,numero_valider_tva
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645,"(False, cle_invalide)"
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,"(True, ok)"
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,"(True, ok)"
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,"(True, ok)"
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88,"(False, cle_invalide)"


In [91]:
#Affichier les motifs de refus 
motifs = df["numero_valider_tva"].apply(lambda resultat: resultat[1])
print(motifs.value_counts())

numero_valider_tva
ok                  6083
cle_invalide        2605
pays_non_couvert    1090
vide                 174
trop_court            48
Name: count, dtype: int64


In [92]:
df_tva_valide = df[df["numero_valider_tva"].str[0] == True]
df_tva_valide.head(5)

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_normalise_tva,numero_valider_tva
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,"(True, ok)"
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,"(True, ok)"
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,"(True, ok)"
5,6,Kappa Logistics AB,DK,DK61530843,2025-01-14,import_fournisseur,DK61530843,"(True, ok)"
7,8,Vertex Industries SARL,SE,SE.314125231601,2025-03-31,reprise_erp,SE314125231601,"(True, ok)"


In [93]:
df_tva_valide[["numero_normalise_tva", "numero_valider_tva"]].head(5)

,numero_normalise_tva,numero_valider_tva
1,PT787327239,"(True, ok)"
2,LU71043234,"(True, ok)"
3,SE751298146001,"(True, ok)"
5,DK61530843,"(True, ok)"
7,SE314125231601,"(True, ok)"


In [94]:
print(len(df_tva_valide["numero_normalise_tva"]))

6083


In [95]:

df_tva_valide["numero_normalise_tva"].duplicated().sum()

np.int64(281)

In [96]:
#supprimer les doublants
#df_tva_valide = df_tva_valide.drop_duplicates(subset=["numero_normalise_tva", "numero_valider_tva"])
#print(len(df_tva_valide))

In [97]:
df_tva_valide = df_tva_valide.drop_duplicates(
    subset=["numero_normalise_tva"]
)
num= len(df_tva_valide)
print("Nombre de lignes dans df_tva_valide :", num)


Nombre de lignes dans df_tva_valide : 5802


In [98]:
#lire le fchier csv 
df_resultats = pd.read_csv("resultats_vies.csv")
# estimer le temps moyen par appel 
temps_moyen = df_resultats["temps_reponse"].mean()

print("Temps moyen par appel :", round(temps_moyen, 2), "secondes")


Temps moyen par appel : 2.56 secondes


In [99]:
df_tva_valide.to_csv("tva_nettoye.csv", index=False, encoding="utf-8")

In [100]:
# Statistiques des temps de réponse
print(df_resultats["temps_reponse"].describe())

count    198.000000
mean       2.555707
std        3.431266
min        0.120000
25%        0.680000
50%        1.305000
75%        2.347500
max       16.630000
Name: temps_reponse, dtype: float64


In [101]:
economises = 10000 - num

print("Nombre d'appels économisés :", economises)

Nombre d'appels économisés : 4198


In [103]:
#Les temps changent selon les pays. Les Pays-Bas sont les plus rapides et le Danemark est le plus lent.
#  La France et la Belgique prennent aussi plus de temps.

In [104]:
#combien de temps prendrait la vérification de tout le référentiel si l'on appelait le service une fois par ligne 
df_resultats.groupby("pays")["temps_reponse"].mean().sort_values()

pays
NL    1.065000
LU    1.078667
IT    1.124375
PT    1.216400
PL    1.445625
SE    2.037059
FI    2.264211
BE    3.033462
FR    3.150435
DK    7.398696
Name: temps_reponse, dtype: float64

In [105]:
temps_total = 10000 * temps_moyen

print("Durée en heures :", round(temps_total / 3600, 2))
print("Durée en minutes :", round(temps_total / 60, 2))

Durée en heures : 7.1
Durée en minutes : 425.95
